# Chapter 11 — Novelty and Outlier Detection

**Nama:** Omar Al Farizy Hutagalung  

**NIM:** 101032300034  

**Mata Kuliah:** Machine Learning  

**Tugas:** Tugas 2 — Code Reproduction & Theoretical Deep-Dive

---

## Objective

Chapter ini membahas pendeteksian anomali, membedakan outlier detection dan novelty detection, serta menggunakan Isolation Forest, Local Outlier Factor, dan One-Class SVM.

## 1. Landasan Teori

Anomaly detection bertujuan menemukan sampel yang berbeda dari pola mayoritas.

- **Outlier detection:** mencari observasi menyimpang dalam data yang tersedia, yang mungkin berisi kontaminasi.
- **Novelty detection:** mempelajari pola data normal dari training set lalu menilai observasi baru.
- **Isolation Forest:** mengisolasi sampel melalui pemisahan acak; anomali cenderung lebih cepat terisolasi.
- **Local Outlier Factor (LOF):** membandingkan kepadatan lokal suatu titik dengan tetangganya.
- **One-Class SVM:** mempelajari batas wilayah data normal.

Anomali tidak selalu berarti kesalahan; anomali juga dapat merupakan kejadian penting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

X_normal, _ = make_blobs(
    n_samples=300, centers=1, cluster_std=0.7, random_state=42
)
rng = np.random.default_rng(42)
X_outliers = rng.uniform(low=-6, high=6, size=(25, 2))
X = np.vstack([X_normal, X_outliers])
y_reference = np.array([1] * len(X_normal) + [-1] * len(X_outliers))

X_scaled = StandardScaler().fit_transform(X)

plt.scatter(X[:, 0], X[:, 1], c=y_reference, s=20)
plt.title("Data sintetis: normal (1) dan anomali (-1)")
plt.show()

## 2. Isolation Forest

Pada API scikit-learn, prediksi `1` berarti inlier dan `-1` berarti outlier. Parameter `contamination` memperkirakan proporsi outlier dalam data.

In [ ]:
isolation = IsolationForest(
    n_estimators=200, contamination=0.07, random_state=42
)
isolation_pred = isolation.fit_predict(X_scaled)

print(pd.Series(isolation_pred).value_counts().rename("Jumlah"))
plt.scatter(X[:, 0], X[:, 1], c=isolation_pred, s=20)
plt.title("Isolation Forest: 1 = inlier, -1 = outlier")
plt.show()

## 3. LOF dan One-Class SVM

LOF pada contoh ini digunakan untuk outlier detection transduktif: `fit_predict` menilai data yang diberikan. One-Class SVM dipakai untuk mempelajari wilayah data normal.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.07)
lof_pred = lof.fit_predict(X_scaled)

one_class = OneClassSVM(nu=0.07, kernel="rbf", gamma="scale")
one_class.fit(X_scaled[:len(X_normal)])
ocsvm_pred = one_class.predict(X_scaled)

comparison = pd.DataFrame({
    "Metode": ["Isolation Forest", "Local Outlier Factor", "One-Class SVM"],
    "Prediksi outlier (-1)": [
        int(np.sum(isolation_pred == -1)),
        int(np.sum(lof_pred == -1)),
        int(np.sum(ocsvm_pred == -1)),
    ],
})
display(comparison)

## 4. Evaluasi dan interpretasi

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Isolation Forest dibandingkan label sintetis:")
print(confusion_matrix(y_reference, isolation_pred, labels=[1, -1]))
print(classification_report(
    y_reference, isolation_pred, labels=[1, -1],
    target_names=["Normal", "Anomali"], zero_division=0
))

## 5. Analisis dan Kesimpulan

- Deteksi anomali dapat menghasilkan false positive dan false negative.
- Parameter contamination atau nu memengaruhi ambang deteksi.
- Scaling sangat penting untuk metode berbasis jarak dan kernel.
- Data sintetis memiliki label acuan sehingga metrik supervisi dapat dihitung, tetapi kasus nyata sering tidak memiliki label anomali yang lengkap.
- Pemilihan metode dan ambang perlu disesuaikan dengan biaya kesalahan di domain aplikasi.

**Kesimpulan:** deteksi anomali adalah alat untuk menandai observasi yang perlu diperiksa, bukan bukti otomatis bahwa suatu observasi salah atau berbahaya.